In [0]:
%sql
MERGE INTO dev_lab.silver.reading_events AS target
USING (
    SELECT
        event_id,
        user_id,
        book_id,
        event_time,
        duration_seconds
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_time DESC
            ) AS rn
        FROM dev_lab.bronze.reading_events
        WHERE event_id IS NOT NULL
          AND user_id IS NOT NULL
          AND book_id IS NOT NULL
          AND event_time IS NOT NULL
          AND duration_seconds >= 0
    ) ranked
    WHERE rn = 1
) AS source
ON target.event_id = source.event_id

WHEN NOT MATCHED THEN INSERT (
    event_id,
    user_id,
    book_id,
    event_time,
    duration_seconds
)
VALUES (
    source.event_id,
    source.user_id,
    source.book_id,
    source.event_time,
    source.duration_seconds
);

In [0]:
%sql
CREATE OR REPLACE TABLE dev_lab.gold.daily_book_metrics
USING DELTA
AS
SELECT
    CAST(event_time AS DATE) AS reading_date,
    book_id,
    COUNT(*) AS total_events,
    COUNT(DISTINCT user_id) AS unique_readers,
    SUM(duration_seconds) AS total_reading_seconds,
    ROUND(AVG(duration_seconds), 2) AS avg_event_duration_seconds
FROM dev_lab.silver.reading_events
GROUP BY
    CAST(event_time AS DATE),
    book_id;

In [0]:
%sql
SELECT *
FROM dev_lab.gold.daily_book_metrics
ORDER BY reading_date, book_id;